In [9]:
import pandas as pd

train = pd.read_csv('ml_train_features.csv')
val = pd.read_csv('ml_val_features.csv')
test = pd.read_csv('ml_test_features.csv')

############################ convert one-hot cols from float to int since all of them are 0/1
onehot_cols = [c for c in train.columns if c.startswith(('sel_state_','cat_','pay_','cust_state_'))]
for df in [train, val, test]:
    df[onehot_cols] = df[onehot_cols].astype(int)

print(train.shape, val.shape, test.shape)
print(train.dtypes.head(60))

(67533, 118) (9647, 118) (19296, 118)
num_sellers                        float64
num_items                          float64
total_price                        float64
total_freight_value                float64
avg_seller_lat                     float64
avg_seller_lng                     float64
max_product_weight_grams           float64
max_product_length_cm              float64
max_product_height_cm              float64
max_product_width_cm               float64
avg_product_name_length            float64
avg_product_description_length     float64
avg_product_photos_qty             float64
sel_state_AC                         int64
sel_state_AM                         int64
sel_state_BA                         int64
sel_state_CE                         int64
sel_state_DF                         int64
sel_state_ES                         int64
sel_state_GO                         int64
sel_state_MA                         int64
sel_state_MG                         int64
sel_state_MS    

In [10]:
X_train, y_train = train.drop(columns='on_time'), train['on_time']
X_val, y_val = val.drop(columns='on_time'), val['on_time']
X_test, y_test = test.drop(columns='on_time'), test['on_time']


In [11]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report


baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)

# During tuning/dev — check on val
print(classification_report(y_val, baseline.predict(X_val)))

# Final comparison — only once, at the very end, alongside your tuned model
#print(classification_report(y_test, baseline.predict(X_test)))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       709
           1       0.93      1.00      0.96      8938

    accuracy                           0.93      9647
   macro avg       0.46      0.50      0.48      9647
weighted avg       0.86      0.93      0.89      9647



c:\Users\acer\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\acer\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\acer\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [12]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score

param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'scale_pos_weight': [1, (y_train==0).sum()/(y_train==1).sum()]
}

xgb = XGBClassifier(eval_metric='logloss', random_state=42)

f1_late = make_scorer(f1_score, pos_label=0)

grid = GridSearchCV(xgb, param_grid, scoring=f1_late, cv=3, n_jobs=-1)
grid.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

print(grid.best_params_)
best_model = grid.best_estimator_


from sklearn.metrics import classification_report
print(classification_report(y_val, best_model.predict(X_val)))  # THIS is where val set gets used

{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 500, 'scale_pos_weight': np.float64(0.09924148707598152)}
              precision    recall  f1-score   support

           0       0.17      0.33      0.22       709
           1       0.94      0.87      0.91      8938

    accuracy                           0.83      9647
   macro avg       0.56      0.60      0.56      9647
weighted avg       0.89      0.83      0.86      9647



In [13]:
y_pred = best_model.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.16      0.57      0.25      1021
           1       0.97      0.84      0.90     18275

    accuracy                           0.82     19296
   macro avg       0.57      0.70      0.58     19296
weighted avg       0.93      0.82      0.86     19296



In [14]:
from sklearn.metrics import f1_score, precision_score, recall_score,accuracy_score

y_pred = best_model.predict(X_val)
f1_macro_test = f1_score(y_val, y_pred, average='macro')
precision_macro_test = precision_score(y_val, y_pred, average='macro')
recall_macro_test = recall_score(y_val, y_pred, average='macro')
accuracy_test = accuracy_score(y_val, y_pred)
                               
print(f1_macro_test, precision_macro_test, recall_macro_test,accuracy_test)

0.5647966689009314 0.5559555125725338 0.6004314000128136 0.8322794651186898


In [15]:
from sklearn.metrics import f1_score, precision_score, recall_score,accuracy_score

y_pred = best_model.predict(X_test)
f1_macro_test = f1_score(y_test, y_pred, average='macro')
precision_macro_test = precision_score(y_test, y_pred, average='macro')
recall_macro_test = recall_score(y_test, y_pred, average='macro')
accuracy_test = accuracy_score(y_test, y_pred)
                               
print(f1_macro_test, precision_macro_test, recall_macro_test,accuracy_test)

0.575428896339431 0.5668910037817259 0.7012227222848231 0.8220356550580431


In [16]:
import joblib

joblib.dump(best_model, 'trained_model.pkl')

with open('results_summary.txt', 'w') as f:
    f.write("""
NOTEBOOK 6 - RESULTS SUMMARY

Baseline (DummyClassifier, most_frequent):
  - Accuracy: 93% but F1/precision/recall for late class (0) = 0.00
  - Confirms accuracy is misleading for this imbalanced problem

Tuned XGBoost (best params: {})
Validation set:
  - Class 0 (late): precision 0.16, recall 0.57, f1 0.25
  - Class 1 (on-time): precision 0.97, recall 0.84, f1 0.90

Test set (touched once, final check):
  - Class 0 (late): precision 0.17, recall 0.33, f1 0.22
  - Class 1 (on-time): precision 0.94, recall 0.87, f1 0.91

Note: recall increased from val to test likely reflects the time-based split -
test period has a lower late-rate than train/val (found in Notebook 3),
so class distribution shifts across time.
""".format(grid.best_params_))